# SVGP for VAR4 (psi) — no_c_alpha (6-D), stanley single-mu

Wraps `svgp_train_cli.py` for interactive use. Edit the hparams below; run cells top-to-bottom.
After hyperparameter search via `hp_search_svgp_ard_cli.py`, plug `NUM_INDUCING` and `SIGMA_Y` from `best_<out_name>.json`.

In [ ]:
import sys, os
sys.path.insert(0, '.')
import numpy as np
import jax
jax.config.update('jax_enable_x64', True)
jax.config.update('jax_platform_name', 'cpu')
import matplotlib.pyplot as plt

from svgp_train_cli import (
    Y_NAMES, load_task_dirs, train_one, per_task_eval, save_npz, save_pkl,
)
print('Loaded svgp_train_cli helpers.')

In [ ]:
# ===== Config =====
VARIDX = 4                         # psi
FEATURE_MODE = 'no_c_alpha'        # 6-D input
MAP_NAME = 'YasMarina'
TS = 0.1
MU = 1.0489
TASK_DIRS = [
    f'../../data/splits/{MAP_NAME}_VAR{VARIDX}_Ts{TS:.3f}_stanley_mu_{MU}',
]

# ===== Hyperparameters (placeholder — replace from HP search results) =====
NUM_INDUCING = 12
SIGMA_Y = 1e-4
INDUCING_INIT = 'random_subset'   # or 'kmeans'
SEED = 42
MAXITER = 100000
VAL_CHECK_EVERY = 10
PATIENCE = 200

# ===== Output =====
OUT_MODELS_DIR = '../../data/models'
OUT_NAME = 'krasser_svgp_psi_stanley_no_c_alpha'

print(f'VARIDX={VARIDX} ({Y_NAMES[VARIDX]}), feature_mode={FEATURE_MODE}')
print(f'task_dirs: {TASK_DIRS}')

In [ ]:
data = load_task_dirs(TASK_DIRS, FEATURE_MODE)
print(f"Combined: train={data['X_train_all'].shape}, val={data['X_val_all'].shape}, input_dim={data['input_dim']}")

In [ ]:
model = train_one(
    data['X_train_all'], data['y_train_all'],
    data['X_val_all'], data['y_val_all'],
    data['yscaler_all'],
    num_inducing=NUM_INDUCING, sigma_y=SIGMA_Y, inducing_init=INDUCING_INIT, seed=SEED,
    maxiter=MAXITER, val_check_every=VAL_CHECK_EVERY, patience=PATIENCE,
)
print(f"\nBest val RMSE: {model['best_val_rmse']:.6f} at iter {model['best_iter']}")

In [ ]:
if model['history']:
    its, rmses = zip(*model['history'])
    plt.figure(figsize=(8, 4))
    plt.plot(its, rmses, label='val RMSE')
    plt.axvline(model['best_iter'], color='red', ls='--', label=f"best @ iter {model['best_iter']}")
    plt.xlabel('L-BFGS-B iter'); plt.ylabel('Val RMSE (orig scale)'); plt.legend(); plt.grid(True)
    plt.title(f"Val RMSE history — VAR{VARIDX} ({Y_NAMES[VARIDX]}), {FEATURE_MODE}")
    plt.tight_layout(); plt.show()

In [ ]:
rows, overall_rmse, overall_mae = per_task_eval(
    model, data['eval_X_per_task'], data['eval_y_per_task'],
    data['yscaler_all'], data['task_names'],
)
print(f"\n{'Task':<55} {'RMSE (orig)':<15} {'MAE (orig)':<15}")
print('-' * 85)
for r in rows:
    print(f"{r['task']:<55} {r['rmse_orig']:<15.6f} {r['mae_orig']:<15.6f}")
print('-' * 85)
print(f"{'OVERALL':<55} {overall_rmse:<15.6f} {overall_mae:<15.6f}")

In [ ]:
import json, _pickle as pickle
from pathlib import Path
Path(OUT_MODELS_DIR).mkdir(parents=True, exist_ok=True)
npz_path = f'{OUT_MODELS_DIR}/{OUT_NAME}_M_{NUM_INDUCING}.npz'
pkl_path = f'{OUT_MODELS_DIR}/{OUT_NAME}_M_{NUM_INDUCING}.pkl'

meta = {
    'varidx': VARIDX, 'y_name': Y_NAMES[VARIDX],
    'feature_mode': FEATURE_MODE,
    'task_dirs': [str(Path(d).resolve()) for d in TASK_DIRS],
    'task_names': data['task_names'],
    'num_inducing': NUM_INDUCING, 'sigma_y': float(SIGMA_Y),
    'inducing_init': INDUCING_INIT, 'seed': SEED,
    'input_dim': data['input_dim'],
    'best_iter': model['best_iter'], 'best_val_rmse': model['best_val_rmse'],
    'eval_overall_rmse': overall_rmse, 'eval_overall_mae': overall_mae,
    'per_task_eval': rows,
}
save_npz(npz_path, model, data['xscaler_all'], data['yscaler_all'], meta)
save_pkl(pkl_path, model, data['xscaler_all'], data['yscaler_all'])
print(f'Saved: {npz_path}')
print(f'Saved: {pkl_path}')